# Exploratory Data Analysis (EDA)
**Dataset:** `mpg_indo.csv` — 398 observasi, target: `konsumsi_bbm` (mpg)

| Section | Isi |
|---|---|
| 1 | Setup & Load Data |
| 2 | Jenis, Bentuk & Tipe Data |
| 3 | Statistik Deskriptif |
| 4 | Missing Value |
| 5 | Deteksi Outlier |
| 6 | Univariate Analysis |
| 7 | Bivariate & Multivariate Analysis |
| 8 | Variance Analysis & Threshold |
| 9 | Summary |
| 10 | Planning Preprocessing |

---
## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy import stats
from scipy.stats import entropy as scipy_entropy
import os, warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)

COLORS  = sns.color_palette('muted')
TARGET  = 'konsumsi_bbm'
FIG_DIR = os.path.join('..', 'reports', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

def save_fig(filename, fig=None, dpi=150):
    path = os.path.join(FIG_DIR, filename if filename.endswith('.png') else filename + '.png')
    (fig if fig is not None else plt).savefig(path, dpi=dpi, bbox_inches='tight')

print('Setup selesai.')

In [ ]:
df = pd.read_csv('../data/raw/mpg_indo.csv')

origin_map = {1: 'Amerika', 2: 'Eropa', 3: 'Asia'}
df['asal_pabrikan']   = df['asal_pabrikan'].map(origin_map)
df['jumlah_silinder'] = df['jumlah_silinder'].astype('object')
df['tahun_rilis']     = df['tahun_rilis'].astype('object')   # kategorikal

print(f'Dataset dimuat: {df.shape[0]} baris × {df.shape[1]} kolom')
df.head()

---
## 2. Jenis, Bentuk & Tipe Data

In [ ]:
print(f'Bentuk: {df.shape[0]} baris × {df.shape[1]} kolom')
df.info()

In [ ]:
num_cols     = df.select_dtypes(include=np.number).columns.tolist()
cat_cols     = df.select_dtypes(include='object').columns.tolist()
num_features = [c for c in num_cols if c != TARGET]

summary_df = pd.DataFrame({
    'Kolom'        : df.columns,
    'Tipe Data'    : df.dtypes.values,
    'Kategori'     : ['Numerik (Target)' if c == TARGET
                      else 'Numerik' if c in num_cols
                      else 'Kategorikal' for c in df.columns],
    'Unique Values': [df[c].nunique() for c in df.columns]
})
display(summary_df)
print(f'Numerik: {len(num_cols)} | Kategorikal: {len(cat_cols)}')

---
## 3. Statistik Deskriptif

In [ ]:
desc = df[num_cols].describe().T
desc['skewness'] = df[num_cols].skew()
desc['kurtosis'] = df[num_cols].kurt()
desc['cv_%']     = (desc['std'] / desc['mean'] * 100).round(2)
print('Statistik Deskriptif — Fitur Numerik:')
display(desc.round(3))

In [ ]:
print('Statistik Deskriptif — Fitur Kategorikal:')
display(df[cat_cols].describe())

for col in cat_cols:
    freq = df[col].value_counts()
    pct  = df[col].value_counts(normalize=True).mul(100).round(1)
    print(f'\nDistribusi [{col}]:')
    display(pd.DataFrame({'Frekuensi': freq, 'Persentase (%)': pct}))

---
## 4. Missing Value

In [ ]:
missing = pd.DataFrame({
    'Jumlah Missing' : df.isnull().sum(),
    'Persentase (%)' : (df.isnull().sum() / len(df) * 100).round(2)
})
missing = missing[missing['Jumlah Missing'] > 0].sort_values('Persentase (%)', ascending=False)

if missing.empty:
    print('Tidak ada missing value.')
else:
    print(f'Missing value ditemukan pada {len(missing)} kolom:')
    display(missing)
    fig, ax = plt.subplots(figsize=(12, 4))
    sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis', ax=ax)
    ax.set_title('Peta Missing Value', fontweight='bold')
    plt.tight_layout()
    save_fig('04_missing_value_heatmap')
    plt.show()

In [ ]:
missing_all = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(missing_all.index, missing_all.values,
               color=['#d9534f' if v > 0 else '#5cb85c' for v in missing_all.values])
ax.set_xlabel('Persentase Missing (%)')
ax.set_title('Persentase Missing Value per Kolom', fontweight='bold')
for bar, val in zip(bars, missing_all.values):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)
plt.tight_layout()
save_fig('04_missing_value_bar')
plt.show()

---
## 5. Deteksi Outlier
Metode: **IQR** (nilai di luar Q1−1.5×IQR ~ Q3+1.5×IQR) dan **Z-Score** (|z| > 3).

In [ ]:
n_cols = len(num_cols)
fig, axes = plt.subplots((n_cols + 2) // 3, 3, figsize=(15, ((n_cols + 2) // 3) * 4))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.boxplot(y=df[col], ax=axes[i], color=COLORS[i % len(COLORS)],
                flierprops=dict(marker='o', markerfacecolor='red', markersize=5, alpha=0.5))
    axes[i].set_title(col, fontweight='bold')
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
fig.suptitle('Boxplot Deteksi Outlier (titik merah = potensi outlier)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
save_fig('05_outlier_boxplot')
plt.show()

In [ ]:
outlier_report = []
for col in num_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR    = Q3 - Q1
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n_iqr    = ((df[col] < lower) | (df[col] > upper)).sum()
    n_zscore = (np.abs(stats.zscore(df[col].dropna())) > 3).sum()
    outlier_report.append({'Fitur': col, 'Batas Bawah IQR': round(lower,2),
                           'Batas Atas IQR': round(upper,2),
                           'Outlier IQR': n_iqr, '% IQR': round(n_iqr/len(df)*100,2),
                           'Outlier Z-Score': n_zscore, '% Z-Score': round(n_zscore/len(df)*100,2)})
outlier_df = pd.DataFrame(outlier_report).set_index('Fitur')
display(outlier_df)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
x, w = np.arange(len(num_cols)), 0.35
b1 = ax.bar(x - w/2, outlier_df['Outlier IQR'],    w, label='IQR',    color=COLORS[0])
b2 = ax.bar(x + w/2, outlier_df['Outlier Z-Score'], w, label='Z-Score', color=COLORS[1])
ax.set_xticks(x)
ax.set_xticklabels(num_cols, rotation=20, ha='right')
ax.set_ylabel('Jumlah Outlier')
ax.set_title('Perbandingan Outlier — IQR vs Z-Score', fontweight='bold')
ax.legend()
ax.bar_label(b1, padding=2, fontsize=9)
ax.bar_label(b2, padding=2, fontsize=9)
plt.tight_layout()
save_fig('05_outlier_iqr_vs_zscore')
plt.show()

---
## 6. Univariate Analysis
### 6.1 Fitur Numerik

In [ ]:
n = len(num_cols)
fig, axes = plt.subplots((n+2)//3, 3, figsize=(15, ((n+2)//3)*4))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    ax = axes[i]
    sns.histplot(df[col].dropna(), kde=True, ax=ax, color=COLORS[i % len(COLORS)], edgecolor='white')
    ax.axvline(df[col].mean(),   color='red',    linestyle='--', linewidth=1.5, label=f'Mean: {df[col].mean():.2f}')
    ax.axvline(df[col].median(), color='orange', linestyle='-.', linewidth=1.5, label=f'Median: {df[col].median():.2f}')
    ax.set_title(f'{col} (Skew: {df[col].skew():.2f})', fontweight='bold')
    ax.legend(fontsize=8)
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
fig.suptitle('Distribusi Fitur Numerik (Histogram + KDE)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
save_fig('06_univariate_numerik_histogram')
plt.show()

In [ ]:
for col in num_cols:
    skew = df[col].skew()
    label = ('Positif kuat' if skew > 1 else 'Positif moderat' if skew > 0.5
             else 'Negatif kuat' if skew < -1 else 'Negatif moderat' if skew < -0.5
             else 'Simetris')
    print(f'  {col:<22}: Skew = {skew:>6.3f}  → {label}')

### 6.2 Fitur Kategorikal

In [ ]:
all_cat  = cat_cols
cat_few  = [c for c in all_cat if df[c].nunique() <= 8]
cat_many = [c for c in all_cat if df[c].nunique() >  8]

if cat_few:
    fig, axes = plt.subplots(1, len(cat_few), figsize=(6*len(cat_few), 5))
    if len(cat_few) == 1: axes = [axes]
    for ax, col in zip(axes, cat_few):
        counts = df[col].value_counts().sort_values(ascending=False)
        bars = ax.bar(counts.index.astype(str), counts.values,
                      color=sns.color_palette('pastel', len(counts)), edgecolor='gray')
        ax.set_title(f'Distribusi: {col}', fontweight='bold')
        ax.set_ylabel('Frekuensi')
        for bar, val in zip(bars, counts.values):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
                    f'{val}\n({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=9)
        ax.set_ylim(0, counts.max()*1.22)
    fig.suptitle('Distribusi Fitur Kategorikal', fontsize=13, fontweight='bold')
    plt.tight_layout()
    save_fig('06_univariate_kategorikal_bar')
    plt.show()

for col in cat_many:
    counts = df[col].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(13, 4))
    bars = ax.bar(counts.index.astype(str), counts.values,
                  color=sns.color_palette('Blues_d', len(counts)), edgecolor='gray')
    ax.set_title(f'Distribusi: {col}', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Frekuensi')
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                f'{val}', ha='center', va='bottom', fontsize=9)
    ax.set_ylim(0, counts.max()*1.18)
    plt.tight_layout()
    save_fig(f'06_univariate_{col}_bar')
    plt.show()

### 6.3 Distribusi Target

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.histplot(df[TARGET], kde=True, ax=axes[0], color=COLORS[0], edgecolor='white')
axes[0].axvline(df[TARGET].mean(),   color='red',    linestyle='--', label=f'Mean: {df[TARGET].mean():.2f}')
axes[0].axvline(df[TARGET].median(), color='orange', linestyle='-.', label=f'Median: {df[TARGET].median():.2f}')
axes[0].set_title(f'Distribusi {TARGET}', fontweight='bold')
axes[0].legend()

sns.boxplot(y=df[TARGET], ax=axes[1], color=COLORS[1],
            flierprops=dict(marker='o', markerfacecolor='red', markersize=6, alpha=0.6))
axes[1].set_title(f'Boxplot {TARGET}', fontweight='bold')

stats.probplot(df[TARGET].dropna(), dist='norm', plot=axes[2])
axes[2].set_title(f'QQ-Plot {TARGET}', fontweight='bold')

fig.suptitle(f'Analisis Distribusi Target: {TARGET}', fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('06_univariate_target_distribusi')
plt.show()

print(f'Skewness: {df[TARGET].skew():.4f} | Kurtosis: {df[TARGET].kurt():.4f}')
print(f'Min/Max : {df[TARGET].min():.1f} / {df[TARGET].max():.1f} mpg')

---
## 7. Bivariate & Multivariate Analysis
### 7.1 Korelasi Fitur Numerik

In [ ]:
corr_matrix = df[num_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=-1, vmax=1, center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Heatmap Korelasi Pearson', fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('07_bivariate_heatmap_korelasi')
plt.show()

In [ ]:
corr_target = corr_matrix[TARGET].drop(TARGET).sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
colors_bar = ['#d9534f' if v < 0 else '#5cb85c' for v in corr_target.values]
bars = ax.barh(corr_target.index, corr_target.values, color=colors_bar, edgecolor='gray')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Korelasi Pearson')
ax.set_title(f'Korelasi Fitur vs Target ({TARGET})', fontweight='bold')
for bar, val in zip(bars, corr_target.values):
    ax.text(val + (0.01 if val >= 0 else -0.01), bar.get_y()+bar.get_height()/2,
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)
plt.tight_layout()
save_fig('07_bivariate_korelasi_vs_target')
plt.show()
print(corr_target.to_string())

In [ ]:
n_feat = len(num_features)
fig, axes = plt.subplots((n_feat+2)//3, 3, figsize=(15, ((n_feat+2)//3)*4))
axes = axes.flatten()
for i, col in enumerate(num_features):
    ax = axes[i]
    ax.scatter(df[col], df[TARGET], alpha=0.4, color=COLORS[i % len(COLORS)], s=30, edgecolors='none')
    valid = df[[col, TARGET]].dropna()
    z = np.polyfit(valid[col], valid[TARGET], 1)
    xline = np.linspace(valid[col].min(), valid[col].max(), 100)
    ax.plot(xline, np.poly1d(z)(xline), 'r--', linewidth=1.5)
    r = valid[col].corr(valid[TARGET])
    ax.set_title(f'{col} vs {TARGET} (r={r:.3f})', fontsize=10, fontweight='bold')
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
fig.suptitle('Scatter Plot Fitur Numerik vs Target', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
save_fig('07_bivariate_scatter_vs_target')
plt.show()

In [ ]:
g = sns.pairplot(df[num_cols], diag_kind='kde', plot_kws={'alpha': 0.4, 's': 20},
                 diag_kws={'color': COLORS[0]}, corner=True)
g.fig.suptitle('Pairplot Fitur Numerik', y=1.01, fontsize=13, fontweight='bold')
save_fig('07_bivariate_pairplot', fig=g.fig)
plt.show()

### 7.2 Fitur Kategorikal vs Target

In [ ]:
for col in all_cat:
    order  = df.groupby(col)[TARGET].median().sort_values(ascending=False).index.tolist()
    rotate = 45 if df[col].nunique() > 6 else 0
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.boxplot(data=df, x=col, y=TARGET, order=order, ax=axes[0], palette='muted',
                flierprops=dict(marker='o', markerfacecolor='red', markersize=5, alpha=0.5))
    axes[0].set_title(f'Boxplot: {col} vs {TARGET}', fontweight='bold')
    axes[0].tick_params(axis='x', rotation=rotate)
    sns.violinplot(data=df, x=col, y=TARGET, order=order, ax=axes[1], palette='pastel', inner='quartile')
    axes[1].set_title(f'Violin: {col} vs {TARGET}', fontweight='bold')
    axes[1].tick_params(axis='x', rotation=rotate)
    fig.suptitle(f'Distribusi {TARGET} per {col}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    save_fig(f'07_bivariate_kategorikal_{col}_vs_target')
    plt.show()
    grp = df.groupby(col)[TARGET].agg(['mean','median','std','count'])
    grp.columns = ['Mean','Median','Std','Count']
    display(grp.sort_values('Median', ascending=False))

In [ ]:
# Rata-rata BBM per asal pabrikan
fig, ax = plt.subplots(figsize=(8, 5))
mean_by_origin = df.groupby('asal_pabrikan')[TARGET].mean().sort_values(ascending=False)
bars = ax.bar(mean_by_origin.index, mean_by_origin.values,
              color=sns.color_palette('muted', len(mean_by_origin)), edgecolor='gray')
ax.set_title('Rata-rata BBM per Asal Pabrikan', fontweight='bold')
ax.set_ylabel('Rata-rata mpg')
for bar, val in zip(bars, mean_by_origin.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{val:.2f}', ha='center', fontsize=10)
plt.tight_layout()
save_fig('07_bivariate_mean_bbm_asal_pabrikan')
plt.show()

In [ ]:
# Rata-rata BBM per jumlah silinder
fig, ax = plt.subplots(figsize=(8, 5))
mean_by_cyl = df.groupby('jumlah_silinder')[TARGET].mean().sort_index()
bars = ax.bar(mean_by_cyl.index.astype(str), mean_by_cyl.values,
              color=sns.color_palette('coolwarm', len(mean_by_cyl)), edgecolor='gray')
ax.set_title('Rata-rata BBM per Jumlah Silinder', fontweight='bold')
ax.set_ylabel('Rata-rata mpg')
for bar, val in zip(bars, mean_by_cyl.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{val:.2f}', ha='center', fontsize=10)
plt.tight_layout()
save_fig('07_bivariate_mean_bbm_jumlah_silinder')
plt.show()

### 7.3 Distribusi Bin Target & Tren Tahun Rilis

In [ ]:
bins   = [0, 15, 20, 25, 30, 100]
labels = ['<15','15-20','20-25','25-30','>30']
df['kategori_bbm'] = pd.cut(df[TARGET], bins=bins, labels=labels)
cat_count = df['kategori_bbm'].value_counts().sort_index()
cat_pct   = df['kategori_bbm'].value_counts(normalize=True).sort_index().mul(100)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bars = axes[0].bar(cat_count.index.astype(str), cat_count.values,
                   color=sns.color_palette('YlOrRd', len(cat_count)), edgecolor='gray')
axes[0].set_title('Distribusi Bin Konsumsi BBM', fontweight='bold')
for bar, (val, pct) in zip(bars, zip(cat_count.values, cat_pct.values)):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                 f'{val}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9)
axes[0].set_ylim(0, cat_count.max()*1.2)
axes[1].pie(cat_pct.values, labels=cat_pct.index.astype(str), autopct='%1.1f%%',
            colors=sns.color_palette('YlOrRd', len(cat_count)),
            startangle=90, wedgeprops={'edgecolor':'white','linewidth':1.5})
axes[1].set_title('Proporsi Bin Konsumsi BBM', fontweight='bold')
fig.suptitle('Cek Imbalanced Target', fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('07_imbalanced_target_bin')
plt.show()
display(pd.DataFrame({'Frekuensi': cat_count, 'Persentase (%)': cat_pct.round(2)}))
df.drop(columns=['kategori_bbm'], inplace=True)

In [ ]:
# Tren BBM per tahun rilis
trend = df.copy()
trend['tahun_rilis_int'] = trend['tahun_rilis'].astype(int)
trend = trend.groupby('tahun_rilis_int')[TARGET].agg(['mean','min','max'])

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(trend.index, trend['mean'], marker='o', color=COLORS[0], linewidth=2, label='Rata-rata')
ax.fill_between(trend.index, trend['min'], trend['max'], alpha=0.15, color=COLORS[0], label='Rentang min-max')
ax.set_title('Tren Konsumsi BBM per Tahun Rilis', fontsize=12, fontweight='bold')
ax.set_xlabel('Tahun Rilis (70=1970)')
ax.set_ylabel('mpg')
ax.legend()
ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
plt.tight_layout()
save_fig('07_tren_bbm_tahun_rilis')
plt.show()

---
## 8. Variance Analysis & Low-Variance Threshold
**CV (%)** = std/mean×100 untuk fitur numerik. **Normalized Entropy** untuk fitur kategorikal.

In [ ]:
CV_THRESHOLD = 10  # %

var_df = pd.DataFrame({'Varians': df[num_cols].var(),
                       'Std Dev': df[num_cols].std(),
                       'Mean'   : df[num_cols].mean()}).round(4)
var_df['CV (%)']       = (var_df['Std Dev'] / var_df['Mean'].abs() * 100).round(2)
var_df['Low Variance?'] = var_df['CV (%)'].apply(lambda x: 'Ya' if x < CV_THRESHOLD else 'Tidak')
print(f'Variance Analysis (Threshold CV = {CV_THRESHOLD}%):')
display(var_df.sort_values('CV (%)'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sorted_cv   = var_df['CV (%)'].sort_values(ascending=False)
cv_colors   = ['#d9534f' if v < CV_THRESHOLD else COLORS[1] for v in sorted_cv.values]
axes[0].barh(var_df['Varians'].sort_values(ascending=False).index,
             var_df['Varians'].sort_values(ascending=False).values, color=COLORS[0], edgecolor='gray')
axes[0].set_title('Varians Absolut', fontweight='bold')
bars = axes[1].barh(sorted_cv.index, sorted_cv.values, color=cv_colors, edgecolor='gray')
axes[1].axvline(x=CV_THRESHOLD, color='red', linestyle='--', linewidth=1.5, label=f'Threshold={CV_THRESHOLD}%')
axes[1].set_title('Coefficient of Variation (%)', fontweight='bold')
axes[1].legend()
fig.suptitle('Analisis Varians Fitur Numerik', fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('08_variance_cv_numerik')
plt.show()

In [ ]:
ENTROPY_THRESHOLD = 0.5

entropy_rows = []
for col in all_cat:
    counts   = df[col].value_counts(normalize=True)
    n_cat    = len(counts)
    ent      = scipy_entropy(counts.values, base=2)
    max_ent  = np.log2(n_cat) if n_cat > 1 else 1
    norm_ent = ent / max_ent
    entropy_rows.append({'Fitur': col, 'Jumlah Kategori': n_cat,
                         'Entropy (bit)': round(ent,4), 'Normalized Entropy': round(norm_ent,4),
                         'Top Kategori (%)': round(counts.iloc[0]*100,2),
                         'Low Diversity?': 'Ya' if norm_ent < ENTROPY_THRESHOLD else 'Tidak'})
entropy_df = pd.DataFrame(entropy_rows).set_index('Fitur')
display(entropy_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ent_vals  = entropy_df['Normalized Entropy'].sort_values()
ent_cols  = ['#d9534f' if v < ENTROPY_THRESHOLD else COLORS[2] for v in ent_vals.values]
axes[0].barh(ent_vals.index, ent_vals.values, color=ent_cols, edgecolor='gray')
axes[0].axvline(ENTROPY_THRESHOLD, color='red', linestyle='--', linewidth=1.5, label=f'Threshold={ENTROPY_THRESHOLD}')
axes[0].set_xlim(0, 1.1)
axes[0].set_title('Normalized Entropy Fitur Kategorikal', fontweight='bold')
axes[0].legend()
top_vals  = entropy_df['Top Kategori (%)'].sort_values(ascending=False)
top_cols  = ['#d9534f' if v > 80 else COLORS[3] for v in top_vals.values]
axes[1].barh(top_vals.index, top_vals.values, color=top_cols, edgecolor='gray')
axes[1].axvline(80, color='orange', linestyle=':', linewidth=1.5, label='Dominasi >80%')
axes[1].set_title('Persentase Kategori Dominan', fontweight='bold')
axes[1].legend()
fig.suptitle('Analisis Keragaman Fitur Kategorikal', fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('08_variance_entropy_kategorikal')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
cv_thresholds  = np.arange(0, 105, 5)
ent_thresholds = np.arange(0, 1.05, 0.05)
axes[0].plot(cv_thresholds, [sum(var_df['CV (%)'] < t) for t in cv_thresholds],
             marker='o', color=COLORS[0], linewidth=2)
axes[0].axvline(CV_THRESHOLD, color='red', linestyle='--', linewidth=1.5, label=f'Threshold={CV_THRESHOLD}%')
axes[0].set_title('Fitur Numerik Di-drop vs Threshold CV', fontweight='bold')
axes[0].set_xlabel('Threshold CV (%)')
axes[0].legend()
axes[1].plot(ent_thresholds, [sum(entropy_df['Normalized Entropy'] < t) for t in ent_thresholds],
             marker='o', color=COLORS[1], linewidth=2)
axes[1].axvline(ENTROPY_THRESHOLD, color='red', linestyle='--', linewidth=1.5, label=f'Threshold={ENTROPY_THRESHOLD}')
axes[1].set_title('Fitur Kategorikal Di-drop vs Threshold Entropy', fontweight='bold')
axes[1].set_xlabel('Threshold Normalized Entropy')
axes[1].legend()
fig.suptitle('Sensitivity Analysis — Threshold Sweep', fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('08_variance_sensitivity_sweep')
plt.show()

---
## 9. Summary

In [ ]:
print('='*60)
print('         RINGKASAN EDA')
print('='*60)
print(f'Dataset         : {df.shape[0]} baris × {df.shape[1]} kolom')
print(f'Fitur numerik   : {len(num_features)} | Kategorikal: {len(all_cat)} | Target: 1')
n_miss = df.isnull().sum().sum()
print(f'Missing value   : {n_miss} ({"ada di kekuatan_mesin" if n_miss > 0 else "bersih"})')
print(f'Target skewness : {df[TARGET].skew():.4f} (right-skewed — perlu transform)')
top_corr = corr_matrix[TARGET].drop(TARGET).abs().sort_values(ascending=False)
print(f'Korelasi tertinggi vs target:')
for feat, val in top_corr.head(3).items():
    direction = 'positif' if corr_matrix[TARGET][feat] > 0 else 'negatif'
    print(f'  {feat:<22}: r={corr_matrix[TARGET][feat]:.3f} ({direction})')
print(f'Multikolinearitas: kapasitas_mesin vs kekuatan_mesin r={corr_matrix.loc["kapasitas_mesin","kekuatan_mesin"]:.3f}')
print('='*60)

---
## 10. Planning Preprocessing

Berdasarkan EDA, strategi preprocessing dan modeling yang akan dijalankan:

```
EDA → Phase 1: Preprocessing → Phase 2: Model Baseline
   → Phase 3: Outlier Handling → Model Improved
   → Phase 4: Feature Engineering → Model Final
```

| Fitur | Missing | Encoding | Transform | Outlier Clip |
|---|---|---|---|---|
| `kapasitas_mesin` | - | - | Yeo-Johnson | Phase 3 |
| `kekuatan_mesin` | Median | - | Yeo-Johnson | Phase 3 |
| `berat_mobil` | - | - | Yeo-Johnson | Phase 3 |
| `akselerasi` | - | - | Yeo-Johnson | Phase 3 |
| `jumlah_silinder` | - | Ordinal (3<4<5<6<8) | - | - |
| `tahun_rilis` | - | Ordinal (70..82) | - | - |
| `asal_pabrikan` | - | One-Hot | - | - |
| `konsumsi_bbm` (target) | - | - | Yeo-Johnson | - |

> **Catatan:** PowerTransformer sklearn dengan `standardize=True` sudah mencakup scaling — tidak perlu StandardScaler terpisah. Target di-transform untuk asumsi residual normal pada model linear, tapi bukan scaling biasa — saat evaluasi prediksi wajib di-inverse transform.